In [94]:
import pandas as pd
import torch
import numpy as np
import os
from torch.utils.data import DataLoader, Dataset
import re
import random
import gc
import json
from collections import Counter, defaultdict
from transformers import (
    BertConfig,
    BertForMaskedLM,
    Trainer,
    TrainingArguments,
    set_seed
)

import torch.nn as nn

from tqdm.auto import tqdm

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

In [95]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [96]:
!unzip /content/bert_preprocess.zip -d ../


Archive:  /content/bert_preprocess.zip
replace ../content/financial_bert_final/vocab.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [97]:
df = pd.read_csv("/content/transactions.tgz",compression='gzip', nrows=8_000_000)
df

KeyboardInterrupt: 

In [ ]:
COLUMN_MAP = {
    "card_transaction.v1.csv": "user",
    "Card": "card",
    "Year": "year",
    "Month": "month",
    "Day": "day",
    "Time": "time",
    "Amount": "amount",
    "Use Chip": "use_chip",
    "Merchant Name": "merchant_name",
    "Merchant City": "merchant_city",
    "Merchant State": "merchant_state",
    "Zip": "zip",
    "MCC": "mcc",
    "Errors?": "errors",
    "Is Fraud?": "is_fraud",
}

df = df.rename(columns=COLUMN_MAP)
df

In [ ]:
date = pd.to_datetime(
    dict(
        year=df["year"],
        month=df["month"],
        day=df["day"]
    )
)
time = pd.to_timedelta(
    df["time"].astype(str) + ":00"
)
df["timestamp"] = date + time
df

In [ ]:
df["amount_numeric"] = df["amount"].astype(str).str.replace("$","").str.replace(",","")
df["amount_numeric"] = pd.to_numeric(df["amount_numeric"])
df.head()

In [ ]:
df = df.sort_values(["user", "timestamp"]).reset_index(drop=True)

In [ ]:
df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = (
    df["timestamp"].dt.dayofweek
)

df["day_of_month"] = (
    df["timestamp"].dt.day
)

df["calendar_month"] = (
    df["timestamp"].dt.month
)

df["previous_time"] = (
    df.groupby("user")["timestamp"]
    .diff()
    .dt
    .total_seconds()
    .div(60)
)
df["previous_time"] = (
    df["previous_time"]
    .fillna(0)
    .clip(lower=0,upper=60*24*30)
)

df

In [ ]:
# df["fraud_label"] = (
#     df["is_fraud"].astype(str)
#     .str.strip()
#     .str.upper()
#     .map({
#         "YES": 1,
#         "NO": 0
#     })

# )

In [ ]:
train_df = df[
    df["timestamp"] < pd.Timestamp("2017-01-01")
].copy()

val_df = df[
    (df["timestamp"] >= pd.Timestamp("2017-01-01"))
    & (df["timestamp"] < pd.Timestamp("2019-01-01"))
].copy()

test_df = df[
    df["timestamp"] >= pd.Timestamp("2019-01-01")
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# ENCODED_PATH = (
#     "/content/encoded_transactions.uint16.mmap"
# )


# encoded_transactions = np.memmap(

#     ENCODED_PATH,

#     dtype=np.uint16,

#     mode="w+",

#     shape=(
#         len(df),
#         TOKENS_PER_TRANSACTION
#     )

# )

In [ ]:
# encoded_transactions.flush()

# print(
#     "Encoded transaction shape:",
#     encoded_transactions.shape
# )

In [ ]:
import gc
del df
gc.collect()

In [ ]:
dataset = [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]

for name, part in dataset:
    print(
        name, part["is_fraud"].value_counts(
            normalize=True
        )
    )

In [ ]:
FOUNDATION_DIR = "/content/financial_bert_final"

VOCAB_PATH = os.path.join(FOUNDATION_DIR,"vocab.json")

PREPROCESSING_PATH = os.path.join(FOUNDATION_DIR,"preprocessing.json")


In [ ]:
with open(VOCAB_PATH, "r") as f:
    token_to_id = json.load(f)


with open(PREPROCESSING_PATH, "r") as f:
    preprocessing = json.load(f)


print("Vocabulary size:", len(token_to_id))

print(
    "Preprocessing keys:",
    preprocessing.keys()
)

In [ ]:
base_model = BertForMaskedLM.from_pretrained(
    "kunley2/FinBERT",
)

In [ ]:
amount_boundaries = np.asarray(
    preprocessing["amount_boundaries"],
    dtype=np.float64
)

delta_boundaries = np.asarray(
    preprocessing["delta_boundaries"],
    dtype=np.float64
)

if "timestamp_boundaries" in preprocessing:

    timestamp_boundaries = np.asarray(
        preprocessing["timestamp_boundaries"],
        dtype=np.float64
    )

else:

    print("timestamp_boundaries missing from preprocessing.json.")

In [ ]:
def fit_quantile_boundaries(values, n_bins):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
    boundaries = np.quantile(values, quantiles)
    boundaries = np.unique(boundaries)
    return boundaries


def apply_quantization(values, boundaries):
    values = np.asarray(values, dtype=np.float64)
    ind = np.digitize(values, boundaries, right=False)
    return ind.astype(dtype=np.int32)


In [ ]:
for name, part in dataset:
    part["amount_bin"] = apply_quantization(
        part["amount_numeric"],
        amount_boundaries
    )


    part["delta_bin"] = apply_quantization(
        part["previous_time"],
        delta_boundaries
    )


    timestamp_seconds = part["timestamp"].astype(np.int64) // 10**9

    part["timestamp_bin"] = apply_quantization(
        timestamp_seconds,
        timestamp_boundaries
    )

In [ ]:
def clean_data(values):
    if pd.isna(values):
        return "NONE"

    values = str(values).strip().upper()
    values = re.sub( r"\s+", "_", values)
    values = values.replace("=","_")
    return values

def clean_zip(values):
    if pd.isna(values):
        return "NONE"

    try:
        number  = float(values)
        if number.is_integer():
            return str(int(number))

    except(TypeError, ValueError):
        pass
    return clean_data(values)

In [ ]:
def transaction_to_tokens(row):

    return [
        "[TXN]",

        f"CARD={clean_data(row.card)}",

        f"TIMESTAMP={row.timestamp_bin}",

        f"HOUR={row.hour}",

        f"DOW={row.day_of_week}",

        f"MONTH={row.calendar_month}",

        f"DOM={row.day_of_month}",

        f"DELTA={row.delta_bin}",

        f"AMOUNT={row.amount_bin}",

        f"CHANNEL={clean_data(row.use_chip)}",

        f"MERCHANT={clean_data(row.merchant_name)}",

        f"CITY={clean_data(row.merchant_city)}",

        f"STATE={clean_data(row.merchant_state)}",

        f"ZIP={clean_zip(row.zip)}",

        f"MCC={clean_data(row.mcc)}",

        f"ERROR={clean_data(row.errors)}",
    ]

In [ ]:
class TransactionTokenizer:

    """
    Tokenizer for structured financial transactions.

    Converts:

        transaction row
            ↓
        field=value tokens
            ↓
        vocabulary IDs

    Uses the SAME vocabulary as the pretrained
    foundation model.
    """


    def __init__(
        self,
        token_to_id,
        include_timestamp=True
    ):

        self.token_to_id = token_to_id

        self.id_to_token = {
            idx: token
            for token, idx
            in token_to_id.items()
        }

        self.include_timestamp = include_timestamp


    @staticmethod
    def clean(value):

        if pd.isna(value):
            return "NONE"

        value = str(value).strip().upper()

        value = re.sub(r"\s+", "_",value)

        value = value.replace("=","_")

        return value


    @staticmethod
    def clean_zip(value):

        if pd.isna(value):
            return "NONE"

        try:

            number = float(value)

            if number.is_integer():

                return str(int(number))

        except (TypeError, ValueError):

            pass


        return TransactionTokenizer.clean(value)


    @staticmethod
    def get_field(token):

        if token.startswith("["):
            return None

        if "=" not in token:
            return None

        return token.split("=",1)[0]


    def tokenize_transaction(self,row):

        tokens = [
            "[TXN]",
            f"CARD={self.clean(row.card)}"
        ]

        if self.include_timestamp:
            tokens.append(f"TIMESTAMP={int(row.timestamp_bin)}")


        tokens.extend([
            f"HOUR={int(row.hour)}",
            f"DOW={int(row.day_of_week)}",
            f"MONTH={int(row.calendar_month)}",
            f"DOM={int(row.day_of_month)}",
            f"DELTA={int(row.delta_bin)}",
            f"AMOUNT={int(row.amount_bin)}",
            f"CHANNEL={self.clean(row.use_chip)}",
            f"MERCHANT={self.clean(row.merchant_name)}",
            f"CITY={self.clean(row.merchant_city)}",
            f"STATE={self.clean(row.merchant_state)}",
            f"ZIP={self.clean_zip(row.zip)}",
            f"MCC={self.clean(row.mcc)}",
            f"ERROR={self.clean(row.errors)}",
        ])


        return tokens

    def encode_token(
        self,
        token
    ):

        # Known token

        if token in self.token_to_id:

            return self.token_to_id[
                token
            ]


        # Try field-specific unknown

        field = self.get_field(token)

        if field is not None:

            unknown_token = (
                f"[UNK_{field}]"
            )


            if unknown_token in self.token_to_id:

                return self.token_to_id[unknown_token]


        # Global unknown

        return self.token_to_id["[UNK]"]



    def encode_transaction(self, row):

        tokens = self.tokenize_transaction(row)

        return [
            self.encode_token(token)
            for token in tokens
        ]


    def decode(self, ids):
        return [
            self.id_to_token.get(
                int(idx),
                "[UNK]"
            )
            for idx in ids

        ]


    @property
    def cls_token_id(self):
        return self.token_to_id["[CLS]"]


    @property
    def sep_token_id(self):
        return self.token_to_id["[SEP]"]

    @property
    def pad_token_id(self):
        return self.token_to_id["[PAD]"]

    @property
    def unk_token_id(self):
        return self.token_to_id["[UNK]"]

    @property
    def txn_token_id(self):
        return self.token_to_id["[TXN]"]


    def __len__(self):
        return len(self.token_to_id)

In [ ]:
tokenizer = TransactionTokenizer(
    token_to_id=token_to_id,
    include_timestamp=True
)

## Following the gpt process

In [ ]:
# for index, row in enumerate(
#     tqdm(
#         df.itertuples(
#             index=False
#         ),

#         total=len(df),
#         desc="Encoding transactions"
#     )
# ):
#     encoded_transactions[
#         index
#     ] = np.asarray(
#         tokenizer.encode_transaction(row),
#         dtype=np.uint16
#     )

In [ ]:
# encoded_transactions.flush()

# print(
#     "Encoded transaction shape:",
#     encoded_transactions.shape
# )

In [ ]:
# encoded_transactions[
#         index
#     ] = np.asarray(

#         tokenizer.encode_transaction(
#             row
#         ),

#         dtype=np.uint16

#     )

In [ ]:
KEEP_COLUMNS = [
    "user",
    "card",

    "timestamp",

    "hour",
    "day_of_week",
    "calendar_month",
    "day_of_month",

    "delta_bin",
    "amount_bin",
    "timestamp_bin",

    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "errors",
    "is_fraud"

]

In [ ]:
all_df = pd.concat(
    [
        train_df[KEEP_COLUMNS],
        val_df[KEEP_COLUMNS],
        test_df[KEEP_COLUMNS]
    ],
    ignore_index=True,
    copy=False
)

In [ ]:
all_df = (
    all_df
    .sort_values(
        [
            "user",
            "timestamp"
        ]
    )
    .reset_index(
        drop=True
    )
)

In [ ]:
sample_row = next(train_df.itertuples(index=False))

sample_tokens = tokenizer.tokenize_transaction(sample_row)

sample_ids = tokenizer.encode_transaction(sample_row)


print(sample_tokens)

print(sample_ids)

print(tokenizer.decode(sample_ids))

In [ ]:
del train_df
del val_df
del test_df

gc.collect()

In [ ]:
all_df["fraud_label"] = (
    all_df["is_fraud"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "YES": 1,
        "NO": 0

    })

)

In [ ]:
TOKENS_PER_TRANSACTION = len(sample_ids)

print(
    "Tokens per transaction:",
    TOKENS_PER_TRANSACTION
)

In [ ]:
ENCODED_PATH = "/content/encoded_transactions.uint16.mmap"

encoded_transactions = np.memmap(
    ENCODED_PATH,
    dtype=np.uint16,
    mode="w+",
    shape=(len(all_df), TOKENS_PER_TRANSACTION)
)

In [37]:
for index, row in enumerate(

    tqdm(
        all_df.itertuples(
            index=False
        ),
        total=len(all_df),
        desc="Encoding transactions"
    )
):

    encoded_transactions[index] = np.asarray(
        tokenizer.encode_transaction(row),
        dtype=np.uint16
    )

Encoding transactions:   0%|          | 0/8000000 [00:00<?, ?it/s]

In [38]:
encoded_transactions.flush()

print(
    "Encoded transaction shape:",
    encoded_transactions.shape
)

Encoded transaction shape: (8000000, 16)


In [39]:
TRANSACTIONS_PER_SEQUENCE = 16

HISTORY_REQUIRED = TRANSACTIONS_PER_SEQUENCE - 1

In [40]:
users = all_df["user"].to_numpy()

In [41]:
user_starts = np.flatnonzero(
    np.r_[True, users[1:] != users[:-1]]
)

In [42]:
user_ends = np.r_[user_starts[1:], len(all_df)]

In [43]:
target_parts = []

for start, end in zip(
    user_starts,
    user_ends
):

    first_target = start + HISTORY_REQUIRED

    if first_target >= end:
        continue

    targets = np.arange(first_target,end, dtype=np.int32)

    target_parts.append(targets)

In [44]:
all_targets = np.concatenate(target_parts)

del target_parts
gc.collect()

print("Eligible fraud examples:", len(all_targets))

Eligible fraud examples: 7989785


In [45]:
timestamps = all_df["timestamp"].to_numpy()

labels = all_df["fraud_label"].to_numpy(dtype=np.uint8)

In [46]:
TRAIN_END = np.datetime64("2017-01-01")

VALIDATION_END = np.datetime64("2019-01-01")

In [47]:
target_timestamps = timestamps[all_targets]

In [48]:
train_mask = target_timestamps < TRAIN_END

validation_mask = (target_timestamps>= TRAIN_END) & (target_timestamps < VALIDATION_END)

test_mask = target_timestamps >= VALIDATION_END

In [49]:
train_targets_all = all_targets[train_mask]
validation_targets = all_targets[validation_mask]
test_targets = all_targets[test_mask]

In [50]:
print("Training targets:", len(train_targets_all))

print("Validation targets:", len(validation_targets))

print("Test targets:", len(test_targets))

Training targets: 6190946
Validation targets: 1123746
Test targets: 675093


In [51]:
SEED = 42

NEGATIVE_TO_POSITIVE_RATIO = 10

rng = np.random.default_rng(SEED)

In [52]:
train_y_all = labels[train_targets_all]

In [53]:
positive_targets = train_targets_all[train_y_all == 1]

negative_targets = train_targets_all[train_y_all == 0]

In [54]:
print("Fraud:", len(positive_targets))

print("Normal:", len(negative_targets))

Fraud: 7953
Normal: 6182993


In [55]:
number_negatives = min(
    len(negative_targets),
    len(positive_targets) * NEGATIVE_TO_POSITIVE_RATIO
)

In [56]:
sampled_negative_targets = rng.choice(
    negative_targets,
    size=number_negatives,
    replace=False
)

In [57]:
train_targets = np.concatenate([positive_targets,sampled_negative_targets])

In [58]:
rng.shuffle(
    train_targets
)

In [59]:
def show_distribution(name, targets):

    y = labels[targets]
    print(f"\n{name}")
    print("Transactions:",len(y))
    print("Fraud:",int(y.sum()))
    print("Fraud rate:", f"{y.mean():.6%}")


show_distribution("TRAIN",train_targets)
show_distribution("VALIDATION",validation_targets)
show_distribution("TEST",test_targets)


TRAIN
Transactions: 87483
Fraud: 7953
Fraud rate: 9.090909%

VALIDATION
Transactions: 1123746
Fraud: 903
Fraud rate: 0.080356%

TEST
Transactions: 675093
Fraud: 627
Fraud rate: 0.092876%


In [60]:
class FraudWindowDataset(Dataset):

    def __init__(
        self,
        targets,
        labels,
        encoded_path,
        number_rows,
        tokens_per_transaction,
        transactions_per_sequence,
        tokenizer
    ):

        self.targets = np.asarray(targets, dtype=np.int32)

        self.labels = labels

        self.encoded_path = encoded_path

        self.number_rows = number_rows

        self.tokens_per_transaction = tokens_per_transaction

        self.transactions_per_sequence = transactions_per_sequence

        self.tokenizer = tokenizer

        self._mmap = None


    def _open_memmap(self):

        if self._mmap is None:
            self._mmap = np.memmap(
                self.encoded_path,
                dtype=np.uint16,
                mode="r",
                shape=(
                    self.number_rows,
                    self.tokens_per_transaction
                )

            )


    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        self._open_memmap()

        target_index = int(self.targets[index])
        start_index = (
            target_index
            - self.transactions_per_sequence
            + 1
        )

        transaction_ids = (
            self._mmap[
                start_index:
                target_index + 1
            ]
            .reshape(-1)
            .astype(
                np.int64
            )
        )

        # [CLS] history/current sequence [SEP]

        input_ids = np.concatenate([
            np.asarray(
                [self.tokenizer.cls_token_id],
                dtype=np.int64
            ),

            transaction_ids,

            np.asarray(
                [self.tokenizer.sep_token_id],
                dtype=np.int64
            )

        ])


        attention_mask = np.ones(
            len(input_ids),
            dtype=np.int64
        )

        label = float(self.labels[target_index])

        return {
            "input_ids":torch.from_numpy(input_ids),
            "attention_mask": torch.from_numpy(attention_mask),

            "labels":torch.tensor(label, dtype=torch.float32)

        }

In [61]:
train_dataset = FraudWindowDataset(
    targets=train_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [62]:
validation_dataset = FraudWindowDataset(
    targets=validation_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [63]:
test_dataset = FraudWindowDataset(
    targets=test_targets,
    labels=labels,
    encoded_path=ENCODED_PATH,
    number_rows=len(all_df),
    tokens_per_transaction=TOKENS_PER_TRANSACTION,
    transactions_per_sequence=TRANSACTIONS_PER_SEQUENCE,
    tokenizer=tokenizer
)

In [64]:
BATCH_SIZE = 32
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory
)

In [65]:
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

In [66]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

In [67]:
class FinancialBERTFraudDetector(nn.Module):

    def __init__(
        self,
        encoder,
        hidden_size,
        head_type="lstm",
        lstm_hidden_size=128,
        dropout=0.2
    ):

        super().__init__()

        self.encoder = encoder

        self.hidden_size = hidden_size

        self.head_type = head_type

        self.encoder_frozen = True

        self.freeze_encoder()

        # LSTM head

        if head_type == "lstm":
            self.lstm = nn.LSTM(
                input_size=hidden_size,
                hidden_size=lstm_hidden_size,
                num_layers=1,
                batch_first=True,
                bidirectional=False

            )

            classifier_input = lstm_hidden_size

        # MLP baselines

        elif head_type in {
            "current_mlp",
            "mean_mlp",
            "cls_mlp"
        }:

            classifier_input = hidden_size


        else:
            raise ValueError(f"Unknown head: {head_type}")

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input,64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64,1)
        )


    # Freeze encoder

    def freeze_encoder(self):

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        self.encoder_frozen = True


    # Fine-tune final BERT layer

    def unfreeze_last_n_layers(self,n=1):

        for parameter in self.encoder.parameters():

            parameter.requires_grad = False

        for layer in self.encoder.encoder.layer[-n:]:

            for parameter in layer.parameters():
                parameter.requires_grad = True

        self.encoder_frozen = False


    def encode(
        self,
        input_ids,
        attention_mask
    ):
        if self.encoder_frozen:

            self.encoder.eval()

            with torch.no_grad():
                outputs = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )


        else:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


        return outputs.last_hidden_state

    def transaction_pool(
        self,
        hidden
    ):

        # Remove CLS and SEP

        hidden = hidden[:, 1:-1, :]

        batch_size = hidden.shape[0]

        hidden = hidden.reshape(
            batch_size,
            TRANSACTIONS_PER_SEQUENCE,
            TOKENS_PER_TRANSACTION,
            self.hidden_size
        )

        field_hidden = hidden[:, :, 1:, :]

        # Mean pooling within EACH transaction

        transaction_embeddings = field_hidden.mean(dim=2)


        return (
            transaction_embeddings
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        hidden = self.encode(
            input_ids,
            attention_mask
        )

        # CLS baseline

        if self.head_type == "cls_mlp":
            representation = hidden[:, 0, :]

        else:
            transaction_embeddings = (
                self.transaction_pool(hidden)
            )

            # Current transaction only

            if self.head_type == "current_mlp":

                representation = transaction_embeddings[:, -1, :]

            # Mean across whole history

            elif self.head_type == "mean_mlp":
                representation = transaction_embeddings.mean(
                    dim=1
                )

            # Temporal LSTM

            elif self.head_type == "lstm":
                sequence_output, _ = self.lstm(transaction_embeddings)

                representation = sequence_output[:, -1, :]

        logits = self.classifier(representation).squeeze(-1)
        return logits

In [68]:
def create_fraud_model(
    head_type,
    fine_tune_last_layers=0
):

    foundation_mlm = BertForMaskedLM.from_pretrained(
        "kunley2/FinBERT"
    )

    model = FinancialBERTFraudDetector(
        encoder=foundation_mlm.bert,
        hidden_size=foundation_mlm.config.hidden_size,
        head_type= head_type,
        lstm_hidden_size=128,
        dropout=0.2
    )

    if fine_tune_last_layers > 0:
        model.unfreeze_last_n_layers(fine_tune_last_layers)

    del foundation_mlm
    gc.collect()

    return model

In [69]:
train_labels_sampled = labels[train_targets]

In [70]:
positive_count = train_labels_sampled.sum()

negative_count = len(train_labels_sampled) - positive_count

In [71]:
POS_WEIGHT = negative_count / positive_count

print("Positive weight:", POS_WEIGHT)

Positive weight: 10.0


In [72]:
@torch.no_grad()
def predict(model, loader):
    model.eval()

    all_probabilities = []

    all_labels = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask)

        probabilities = torch.sigmoid(logits).cpu().numpy()

        all_probabilities.append(probabilities)

        all_labels.append(batch["labels"].numpy())


    return (
        np.concatenate(all_labels),
        np.concatenate(all_probabilities)
    )

In [73]:
def find_best_threshold( y_true, probabilities):

    precision, recall, thresholds = (
        precision_recall_curve(y_true, probabilities)
    )


    precision = precision[:-1]

    recall = recall[:-1]


    f1 = (
        2 * precision * recall /
        np.maximum(precision + recall, 1e-12)
    )


    best = np.argmax(f1)

    return {
        "threshold": float(thresholds[best]),
        "precision": float(precision[best]),
        "recall": float(recall[best]),
        "f1": float(f1[best])
    }

In [74]:
def train_model(
    model,
    epochs=6,
    head_lr=3e-4,
    encoder_lr=1e-5,
    patience=2
):

    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            POS_WEIGHT,
            dtype=torch.float32,
            device=device
        )
    )


    head_parameters = []

    encoder_parameters = []


    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        if name.startswith("encoder."):
            encoder_parameters.append(parameter)
        else:
            head_parameters.append(parameter)


    parameter_groups = [
        {
            "params": head_parameters,
            "lr": head_lr
        }
    ]

    if encoder_parameters:
        parameter_groups.append({
            "params": encoder_parameters,
            "lr": encoder_lr
        })


    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=1e-4
    )


    best_ap = -1

    best_state = None

    no_improvement = 0


    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0
        examples = 0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch}"
        )


        for batch in progress:

            input_ids = batch["input_ids"].to(device)


            attention_mask = batch["attention_mask"].to(device)

            y = batch["labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(input_ids, attention_mask)

            loss = criterion(logits, y)

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                [
                    p for p in model.parameters() if p.requires_grad
                ],
                1.0
            )

            optimizer.step()

            size = input_ids.size(0)

            running_loss += loss.item() * size

            examples += size

            progress.set_postfix(
                loss=(
                    running_loss
                    / examples
                )
            )


        # Validation

        y_val, p_val = predict(model, validation_loader)


        val_ap = average_precision_score(y_val, p_val)

        val_auc = roc_auc_score(y_val, p_val)

        threshold_info = find_best_threshold(y_val, p_val)


        print(f"\nEpoch {epoch}")

        print(f"Validation AP: "f"{val_ap:.6f}")

        print(f"Validation ROC-AUC: "f"{val_auc:.6f}")

        print("Best validation:",threshold_info)

        if val_ap > best_ap:

            best_ap = val_ap


            best_state = {

                key:
                    value.detach()
                    .cpu()
                    .clone()

                for key, value
                in model
                .state_dict()
                .items()

            }
            no_improvement = 0

        else:
            no_improvement += 1

            if no_improvement >= patience:
                print("Early stopping.")
                break


    model.load_state_dict(best_state)

    return model

In [75]:
def evaluate_model(model):

    y_val, p_val = predict(model,validation_loader)

    threshold_info = (
        find_best_threshold(
            y_val,
            p_val
        )
    )

    threshold = threshold_info["threshold"]

    y_test, p_test = predict(model, test_loader)

    predictions = (
        p_test >= threshold
    ).astype(np.uint8)


    ap = (
        average_precision_score(
            y_test,
            p_test
        )
    )


    roc_auc = (
        roc_auc_score(
            y_test,
            p_test
        )
    )


    precision = (
        precision_score(
            y_test,
            predictions,
            zero_division=0
        )
    )


    recall = (
        recall_score(
            y_test,
            predictions,
            zero_division=0
        )
    )


    f1 = (
        f1_score(
            y_test,
            predictions,
            zero_division=0
        )
    )


    cm = confusion_matrix(y_test, predictions)


    result = {
        "PR_AUC_AP": ap,
        "ROC_AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Threshold": threshold
    }


    print(result)


    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            digits=5,
            zero_division=0
        )
    )

    return result

## MLP

In [76]:
current_model = create_fraud_model(head_type="current_mlp")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [77]:
current_model = train_model(
    current_model
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.034541
Validation ROC-AUC: 0.950663
Best validation: {'threshold': 0.9725497364997864, 'precision': 0.059637404580152674, 'recall': 0.13842746400885936, 'f1': 0.08336112037345782}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.065172
Validation ROC-AUC: 0.965120
Best validation: {'threshold': 0.9903503060340881, 'precision': 0.1395856052344602, 'recall': 0.14174972314507198, 'f1': 0.14065934065934066}


Epoch 6:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 6
Validation AP: 0.051708
Validation ROC-AUC: 0.967812
Best validation: {'threshold': 0.9865729808807373, 'precision': 0.08371559633027523, 'recall': 0.16168327796234774, 'f1': 0.11031356252361164}


In [78]:
current_results = evaluate_model(current_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.05133919909246689), 'ROC_AUC': np.float64(0.9656029115379179), 'Precision': 0.11275415896487985, 'Recall': 0.09728867623604466, 'F1': 0.10445205479452055, 'Threshold': 0.9903503060340881}

Confusion Matrix:
[[673986    480]
 [   566     61]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99916   0.99929   0.99922    674466
         1.0    0.11275   0.09729   0.10445       627

    accuracy                        0.99845    675093
   macro avg    0.55596   0.54829   0.55184    675093
weighted avg    0.99834   0.99845   0.99839    675093



In [79]:
del current_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Mean Pooling

In [80]:
mean_model = create_fraud_model(head_type="mean_mlp")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [81]:
mean_model = train_model(mean_model)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.004841
Validation ROC-AUC: 0.768036
Best validation: {'threshold': 0.9193549156188965, 'precision': 0.014205986808726534, 'recall': 0.09302325581395349, 'f1': 0.02464788732394366}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.012799
Validation ROC-AUC: 0.787019
Best validation: {'threshold': 0.9859811067581177, 'precision': 0.06424581005586592, 'recall': 0.02547065337763012, 'f1': 0.03647898493259318}


Epoch 6:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 6
Validation AP: 0.013410
Validation ROC-AUC: 0.794198
Best validation: {'threshold': 0.9782804250717163, 'precision': 0.029321930360415395, 'recall': 0.053156146179401995, 'f1': 0.03779527559055118}


In [82]:
mean_results = (
    evaluate_model(mean_model)
)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.00539837096981401), 'ROC_AUC': np.float64(0.7645871511389215), 'Precision': 0.022813688212927757, 'Recall': 0.028708133971291867, 'F1': 0.025423728813559324, 'Threshold': 0.9782804250717163}

Confusion Matrix:
[[673695    771]
 [   609     18]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99910   0.99886   0.99898    674466
         1.0    0.02281   0.02871   0.02542       627

    accuracy                        0.99796    675093
   macro avg    0.51096   0.51378   0.51220    675093
weighted avg    0.99819   0.99796   0.99807    675093



In [83]:
del mean_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## LSTM

In [84]:
lstm_model = create_fraud_model(head_type="lstm")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [85]:
lstm_model = train_model(
    lstm_model
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.054880
Validation ROC-AUC: 0.956224
Best validation: {'threshold': 0.9970681071281433, 'precision': 0.11017838405036726, 'recall': 0.11627906976744186, 'f1': 0.11314655172413791}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.067973
Validation ROC-AUC: 0.941359
Best validation: {'threshold': 0.9803158640861511, 'precision': 0.11194731890874883, 'recall': 0.13178294573643412, 'f1': 0.12105798575788405}


Epoch 5:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.075027
Validation ROC-AUC: 0.948479
Best validation: {'threshold': 0.985145092010498, 'precision': 0.09296353364149974, 'recall': 0.20044296788482835, 'f1': 0.1270175438596491}
Early stopping.


In [86]:
lstm_results = evaluate_model(lstm_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.0875079773916334), 'ROC_AUC': np.float64(0.9617648063534375), 'Precision': 0.13355317394888705, 'Recall': 0.2583732057416268, 'F1': 0.17608695652173914, 'Threshold': 0.962584376335144}

Confusion Matrix:
[[673415   1051]
 [   465    162]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99931   0.99844   0.99888    674466
         1.0    0.13355   0.25837   0.17609       627

    accuracy                        0.99775    675093
   macro avg    0.56643   0.62841   0.58748    675093
weighted avg    0.99851   0.99775   0.99811    675093



In [87]:
torch.save(
    lstm_model.state_dict(),
    "/content/frozen_bert_lstm_fraud.pt"
)

In [88]:
del lstm_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Tuning only the BERT final layer

In [89]:
finetune_model = (
    create_fraud_model(
        head_type="lstm",
        fine_tune_last_layers=1
    )
)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [90]:
trainable = sum(

    p.numel()
    for p in
    finetune_model.parameters()
    if p.requires_grad
)


total = sum(

    p.numel()
    for p in
    finetune_model.parameters()
)

print(f"Trainable: {trainable:,}")

print(f"Total: {total:,}")

print(f"Percentage: "f"{100 * trainable / total:.2f}%")

Trainable: 338,689
Total: 3,812,609
Percentage: 8.88%


In [91]:
finetune_model = train_model(
    finetune_model,
    epochs=5,
    head_lr=1e-4,
    encoder_lr=1e-5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.091205
Validation ROC-AUC: 0.966933
Best validation: {'threshold': 0.968789279460907, 'precision': 0.13825363825363826, 'recall': 0.14728682170542637, 'f1': 0.14262734584450404}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.118488
Validation ROC-AUC: 0.977352
Best validation: {'threshold': 0.9104887843132019, 'precision': 0.1385154880187025, 'recall': 0.26245847176079734, 'f1': 0.1813312930374904}


Epoch 5:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.126223
Validation ROC-AUC: 0.980508
Best validation: {'threshold': 0.9719676971435547, 'precision': 0.14701378254211334, 'recall': 0.31893687707641194, 'f1': 0.20125786163522014}


In [92]:
finetune_results = evaluate_model(finetune_model)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.11612260067864896), 'ROC_AUC': np.float64(0.9844043861013543), 'Precision': 0.1607717041800643, 'Recall': 0.3189792663476874, 'F1': 0.21378941742383753, 'Threshold': 0.9719676971435547}

Confusion Matrix:
[[673422   1044]
 [   427    200]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99937   0.99845   0.99891    674466
         1.0    0.16077   0.31898   0.21379       627

    accuracy                        0.99782    675093
   macro avg    0.58007   0.65872   0.60635    675093
weighted avg    0.99859   0.99782   0.99818    675093



In [93]:
comparison = pd.DataFrame([
    {

        "Model": "BERT Current Transaction + MLP",
        **current_results
    },
    {
        "Model": "BERT Whole-Sequence Mean + MLP",
        **mean_results
    },
    {
        "Model": "Frozen BERT + Transaction LSTM",
        **lstm_results
    },
    {

        "Model": "Last BERT Layer FT + LSTM",
        **finetune_results
    }
])

In [98]:
torch.cuda.empty_cache()

## Finetuning Two layers

In [99]:
finetune_model2 = (
    create_fraud_model(
        head_type="lstm",
        fine_tune_last_layers=2
    )
)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

In [100]:
trainable = sum(
    p.numel()

    for p in
    finetune_model2.parameters()
    if p.requires_grad

)

total = sum(

    p.numel()

    for p in
    finetune_model2.parameters()
)

print(f"Trainable: {trainable:,}")

print(f"Total: {total:,}")

print(f"Percentage: "f"{100 * trainable / total:.2f}%")

Trainable: 536,961
Total: 3,812,609
Percentage: 14.08%


In [101]:
finetune_model2 = train_model(
    finetune_model2,
    epochs=5,
    head_lr=1e-4,
    encoder_lr=1e-5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.186211
Validation ROC-AUC: 0.984650
Best validation: {'threshold': 0.9716575145721436, 'precision': 0.19516509433962265, 'recall': 0.36655592469545956, 'f1': 0.2547133512889573}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]

...

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 4
Validation AP: 0.164328
Validation ROC-AUC: 0.977123
Best validation: {'threshold': 0.9834741950035095, 'precision': 0.20534069981583794, 'recall': 0.2469545957918051, 'f1': 0.22423328305681248}


Epoch 5:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 5
Validation AP: 0.208714
Validation ROC-AUC: 0.979793
Best validation: {'threshold': 0.9832730293273926, 'precision': 0.22162576687116564, 'recall': 0.32004429678848284, 'f1': 0.26189397371998185}
Early stopping.


In [102]:
finetune_results2 = evaluate_model(finetune_model2)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.21015327977366954), 'ROC_AUC': np.float64(0.9847974160345959), 'Precision': 0.28851540616246496, 'Recall': 0.32854864433811803, 'F1': 0.30723340790454884, 'Threshold': 0.9895845055580139}

Confusion Matrix:
[[673958    508]
 [   421    206]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99938   0.99925   0.99931    674466
         1.0    0.28852   0.32855   0.30723       627

    accuracy                        0.99862    675093
   macro avg    0.64395   0.66390   0.65327    675093
weighted avg    0.99872   0.99862   0.99867    675093



# Finetunning for longer period

In [103]:
torch.cuda.empty_cache()

In [104]:
finetune_model2 = train_model(
    finetune_model2,
    epochs=12,
    head_lr=1e-4,
    encoder_lr=1e-5
)

Epoch 1:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 1
Validation AP: 0.204742
Validation ROC-AUC: 0.979279
Best validation: {'threshold': 0.9845960736274719, 'precision': 0.23850823937554205, 'recall': 0.30454042081949056, 'f1': 0.26750972762645914}


Epoch 2:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 2
Validation AP: 0.122340
Validation ROC-AUC: 0.963181
Best validation: {'threshold': 0.9848233461380005, 'precision': 0.18201997780244172, 'recall': 0.18161683277962348, 'f1': 0.18181818181818182}


Epoch 3:   0%|          | 0/2734 [00:00<?, ?it/s]

  0%|          | 0/35118 [00:00<?, ?it/s]


Epoch 3
Validation AP: 0.154441
Validation ROC-AUC: 0.962072
Best validation: {'threshold': 0.9963177442550659, 'precision': 0.22388059701492538, 'recall': 0.2159468438538206, 'f1': 0.21984216459977451}
Early stopping.


In [105]:
finetune_results2 = evaluate_model(finetune_model2)

  0%|          | 0/35118 [00:00<?, ?it/s]

  0%|          | 0/21097 [00:00<?, ?it/s]

{'PR_AUC_AP': np.float64(0.20191825428943916), 'ROC_AUC': np.float64(0.981265474259698), 'Precision': 0.26631853785900783, 'Recall': 0.3253588516746411, 'F1': 0.2928930366116296, 'Threshold': 0.9845960736274719}

Confusion Matrix:
[[673904    562]
 [   423    204]]

Classification Report:
              precision    recall  f1-score   support

         0.0    0.99937   0.99917   0.99927    674466
         1.0    0.26632   0.32536   0.29289       627

    accuracy                        0.99854    675093
   macro avg    0.63285   0.66226   0.64608    675093
weighted avg    0.99869   0.99854   0.99861    675093

